# Exploratory Data Analysis Exercise with Pandas and Matplotlib

In this exercise, you are responsible for devleoping a data pipeline to ingest and analyze multi-state streamflow records from CSV files in our Canvas Class. This exercise will directly assist with HW #1. Filepath for the data:

    files -> Data -> NWIS_Streaflow -> <STATE>

You will download the data from Canvas and load it into a folder you create called "streamflow_data". Once within the repo, you will load the data into this python notebook and perform exploratory data analysis. After performing data cleaning and time-series alignment with Pandas, you will transition develop Matplotlib visualizations. The core of the assignment emphasizes the Matplotlib philosophy, challenging you to use powerful operators to link, overlay, and explore discharge trends across Idaho, Utah, and Wyoming.

The [USGS NWIS Mapper](https://apps.usgs.gov/nwismapper/) provides interactive mapping to locate sites and repective metadata.

## Task 1: Select, download, and bring the data into your notebook session

Use the [USGS NWIS Mapper](https://apps.usgs.gov/nwismapper/) to locate one site below a reservoir,  one site in a headwater catchment, and one site near a rivers terminus to the Great Salt Lake. Using this siteid, find the site data in the Canvas NWIS_Streamflow data folder, download it to your computer, then upload it to this repo into a folder named "streamflow_data". In the code block below, load the data into a Pandas DataFrame and inspect it as we previously did in the Pandas exercises (.head(), .describe()). Write down what you notice. Remove any outliers NaN values, and -999.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Three Utah sites:
#   10215900 = small headwater tributary (upper watershed)
#   10136500 = Weber River below Echo/Rockport Reservoir
#   10171000 = Jordan River near SLC, drains toward GSL
data_dir = 'streamflow_data'
files = {
    'Headwater':       os.path.join(data_dir, '10215900_1980_2020.csv'),
    'Below Reservoir': os.path.join(data_dir, '10136500_1980_2020.csv'),
    'GSL Terminus':    os.path.join(data_dir, '10171000_1980_2020.csv'),
}

dfs = {}
for label, path in files.items():
    df = pd.read_csv(path, parse_dates=['Datetime'], index_col='Datetime')
    print(f"\n--- {label} (USGS {os.path.basename(path).split('_')[0]}) ---")
    # Remove -999 sentinel values, coerce to numeric, drop NaN/negatives
    df['USGS_flow'] = df['USGS_flow'].replace(-999, pd.NA)
    df['USGS_flow'] = pd.to_numeric(df['USGS_flow'], errors='coerce')
    df = df[df['USGS_flow'] >= 0].dropna(subset=['USGS_flow'])
    print(df['USGS_flow'].describe())
    print(f'Rows after cleaning: {len(df)}')
    dfs[label] = df

dfs['Headwater'].head()


--- Headwater (USGS 10215900) ---
count    10128.000000
mean        25.135896
std         44.185378
min          2.002083
25%          4.984375
50%          8.061198
75%         22.067283
max        576.693850
Name: USGS_flow, dtype: float64
Rows after cleaning: 10128

--- Below Reservoir (USGS 10136500) ---
count    11682.000000
mean       406.207310
std        513.744096
min         17.333334
25%        105.866670
50%        287.151045
75%        424.942712
max       4865.521000
Name: USGS_flow, dtype: float64
Rows after cleaning: 11682

--- GSL Terminus (USGS 10171000) ---
count    11770.000000
mean       128.355038
std         39.204765
min          1.941667
25%        109.854164
50%        128.130210
75%        150.000000
max        317.166660
Name: USGS_flow, dtype: float64
Rows after cleaning: 11770


,USGS_flow,variable,USGS_ID,measurement_unit,qualifiers,series
Datetime,,,,,,
1983-10-01,25.400000,streamflow,10215900,ft3/s,['A'],0
1986-10-01,10.647058,streamflow,10215900,ft3/s,"['A', '[92]']",0
1986-10-02,12.791667,streamflow,10215900,ft3/s,"['A', '[92]']",0
1986-10-03,13.208333,streamflow,10215900,ft3/s,"['A', '[92]']",0
1986-10-04,11.791667,streamflow,10215900,ft3/s,"['A', '[92]']",0


## Task 2: Slicing and Dicing

We are interested in examining the data from 2000-2010. Slice the data accordingly and save it to a new Pandas DataFrame.

In [2]:
# Slice each DataFrame to 2000-2010
dfs_slice = {}
for label, df in dfs.items():
    sliced = df['2000':'2010'].copy()
    dfs_slice[label] = sliced
    print(f"{label}: {len(sliced)} rows | {sliced.index.min().date()} -> {sliced.index.max().date()}")

dfs_slice['Headwater'].head()

Headwater: 3406 rows | 2000-01-01 -> 2010-12-31
Below Reservoir: 3897 rows | 2000-01-01 -> 2010-12-31
GSL Terminus: 3945 rows | 2000-01-01 -> 2010-12-31


,USGS_flow,variable,USGS_ID,measurement_unit,qualifiers,series
Datetime,,,,,,
2000-01-01,4.6875,streamflow,10215900,ft3/s,"['A', '[91]']",0
2000-01-02,4.6000,streamflow,10215900,ft3/s,"['A', '[91]']",0
2000-01-03,4.6000,streamflow,10215900,ft3/s,"['A', '[91]']",0
2000-01-04,4.6000,streamflow,10215900,ft3/s,"['A', '[91]']",0
2000-01-05,4.6000,streamflow,10215900,ft3/s,"['A', '[91]']",0


## Task 3: Create plots for each DataFrame using the df.plot() function

Use the built in functionality of Pandas to plot the time series of each stream.

In [3]:
# Task 3: Plot each site using the built-in df.plot() method
colors = {'Headwater': '#2166ac', 'Below Reservoir': '#d73027', 'GSL Terminus': '#1a9850'}

for label, df in dfs_slice.items():
    ax = df['USGS_flow'].plot(
        figsize=(12, 4),
        title=f'Streamflow 2000-2010: {label}',
        color=colors[label]
    )
    ax.set_xlabel('Date')
    ax.set_ylabel('Streamflow (ft³/s)')
    ax.legend([label])
    plt.tight_layout()
    plt.show()

/scratch/local/u1437843/1197179/ipykernel_53302/3070137048.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/scratch/local/u1437843/1197179/ipykernel_53302/3070137048.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/scratch/local/u1437843/1197179/ipykernel_53302/3070137048.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Task 4: Join/Merge Pandas DataFrames

Create a single dataframe named All_Streams and combine all streamflow monitoring data into this dataframe. Hint, set your index to the date. Create custom labels for each monitoring station location to communicate there location within the watershed (e.g, headwater, below reservoir, GSL Terminus). Print the dataframe.head() to demonstrate that is complete.

In [4]:
# Task 4: Merge all three sites into a single DataFrame indexed by date
All_Streams = pd.concat(
    [df['USGS_flow'].rename(label) for label, df in dfs_slice.items()],
    axis=1
)
All_Streams.index.name = 'Date'

print(All_Streams.head())
print(f"\nShape: {All_Streams.shape}")
print(f"Columns: {All_Streams.columns.tolist()}")
print(f"Date range: {All_Streams.index.min().date()} -> {All_Streams.index.max().date()}")

            Headwater  Below Reservoir  GSL Terminus
Date                                                
2000-01-01     4.6875        131.85417     133.30208
2000-01-02     4.6000        260.55210     133.51042
2000-01-03     4.6000        278.80210     134.62500
2000-01-04     4.6000        235.40625     134.09375
2000-01-05     4.6000        191.84375     134.79167

Shape: (4018, 3)
Columns: ['Headwater', 'Below Reservoir', 'GSL Terminus']
Date range: 2000-01-01 -> 2010-12-31


## Task 5: Demonstrate your Prowess with Matplotlib

Create a Four separate figures with all three stream on them:

* Figure 1 should be a single plot with all three stream  monitoring locations
* Figure 2 should be a single figure with subplots for each stream monitoring location. The subplots should be 2 rows and 2 columns
* Figure 3 should be a single figure with subplots for each stream monitoring location. The subplots should be 3 rows and 1 column 
* Figure 3 should be a single figure with subplots for each stream monitoring location. The subplots should be 1 row and 3 columns

Make sure your plots have the correct axes, labeled axes, a title, a legend. Create custom labels for each monitoring station location to communicate there location within the watershed (e.g, headwater, below reservoir, GSL Terminus).

In [5]:
colors = {'Headwater': '#2166ac', 'Below Reservoir': '#d73027', 'GSL Terminus': '#1a9850'}

# Figure 1: All three streams on a single plot
fig, ax = plt.subplots(figsize=(14, 5))
for col in All_Streams.columns:
    ax.plot(All_Streams.index, All_Streams[col], color=colors[col], lw=1, alpha=0.85, label=col)
ax.set_xlabel('Date')
ax.set_ylabel('Streamflow (ft³/s)')
ax.set_title('Utah Streamflow 2000-2010: All Monitoring Locations')
ax.legend()
plt.tight_layout()
plt.show()

# Figure 2: 2x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes_flat = axes.flatten()
for i, col in enumerate(All_Streams.columns):
    axes_flat[i].plot(All_Streams.index, All_Streams[col], color=colors[col], lw=1)
    axes_flat[i].set_title(col)
    axes_flat[i].set_xlabel('Date')
    axes_flat[i].set_ylabel('Streamflow (ft³/s)')
    axes_flat[i].legend([col])
axes_flat[-1].set_visible(False)
fig.suptitle('Utah Streamflow 2000-2010 (2x2 Subplots)', fontsize=14)
plt.tight_layout()
plt.show()

# Figure 3: 3x1 subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for i, col in enumerate(All_Streams.columns):
    axes[i].plot(All_Streams.index, All_Streams[col], color=colors[col], lw=1)
    axes[i].set_title(col)
    axes[i].set_ylabel('Streamflow (ft³/s)')
    axes[i].legend([col])
axes[-1].set_xlabel('Date')
fig.suptitle('Utah Streamflow 2000-2010 (3x1 Subplots)', fontsize=14)
plt.tight_layout()
plt.show()

# Figure 4: 1x3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(All_Streams.columns):
    axes[i].plot(All_Streams.index, All_Streams[col], color=colors[col], lw=1)
    axes[i].set_title(col)
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Streamflow (ft³/s)')
    axes[i].legend([col])
    axes[i].tick_params(axis='x', rotation=30)
fig.suptitle('Utah Streamflow 2000-2010 (1x3 Subplots)', fontsize=14)
plt.tight_layout()
plt.show()

/scratch/local/u1437843/1197179/ipykernel_53302/2425284059.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/scratch/local/u1437843/1197179/ipykernel_53302/2425284059.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/scratch/local/u1437843/1197179/ipykernel_53302/2425284059.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/scratch/local/u1437843/1197179/ipykernel_53302/2425284059.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
